In [1]:
# 1. Install dependencies
!pip install dm_control

# 2. Clone TD-MPC into the tdmpc/ folder inside your project
%cd /content
!git clone https://github.com/odykon/td-mpc_o2_local

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 114.2 MB/s eta 0:00:00
/content
Cloning into 'td-mpc_o2_local'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 130 (delta 45), reused 42 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.51 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [2]:
import sys
sys.path.insert(0, '/content/td-mpc_o2_local')
sys.path.insert(0, '/content/td-mpc_o2_local/tdmpc/src')

In [3]:
from pathlib import Path
from cfg import parse_cfg
from omegaconf import OmegaConf
from env import make_env

def make_cfg(task, cfg_path=Path('/content/td-mpc_o2_local/tdmpc/cfgs')):
    import sys
    # clear any previous task argument
    sys.argv = [arg for arg in sys.argv if not arg.startswith('task=')]
    sys.argv += [f'task={task}']
    return parse_cfg(cfg_path)

# load the base config
cfg = make_cfg('cheetah-run')


for key, value in cfg.items():
    print(f"{key}: {value}")



/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


task: cheetah-run
modality: state
action_repeat: 4
discount: 0.99
episode_length: 250
train_steps: 125000
iterations: 6
num_samples: 512
num_elites: 64
mixture_coef: 0.05
min_std: 0.05
temperature: 0.5
momentum: 0.1
batch_size: 512
max_buffer_size: 1000000
horizon: 5
reward_coef: 0.5
value_coef: 0.1
consistency_coef: 2
rho: 0.5
kappa: 0.1
lr: 0.001
std_schedule: linear(0.5, 0.05, 25000)
horizon_schedule: linear(1, 5, 25000)
per_alpha: 0.6
per_beta: 0.4
grad_clip_norm: 10
seed_steps: 5000
update_freq: 2
tau: 0.01
enc_dim: 256
mlp_dim: 512
latent_dim: 50
use_wandb: False
wandb_project: none
wandb_entity: none
seed: 1
exp_name: default
eval_freq: 20000
eval_episodes: 10
save_video: False
save_model: False
-f: True
/root/: {'local/share/jupyter/runtime/kernel-314fac82-4bf3-46ce-8646-c03db3d660c8': {'json': None}}
task_title: Cheetah Run
device: cuda


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
ar = cfg.action_repeat  # already set from task yaml

cfg = OmegaConf.merge(cfg, OmegaConf.create({
    # planning
    'num_samples':  64,
    'num_elites':   16,
    'iterations':   5,
    'momentum':     0.1,

    # decoder
    'decoder_init':        True,
    'latent_action_dim':   128,
    'use_latent_state':    True,
    'dcem_batch_size':     64,
    'decoder_updates':     70,
    'lml_temperature':     10.0,

    # TOLD learning
    'batch_size':    512,
    'told_updates':  500,

    # evaluation
    'eval_episodes': 0,
    'video_mode':    'none',

    # training — divided by action_repeat so wall-clock steps are consistent
    'train_steps':  20000 // ar,
    'seed_steps':   0,

    # schedules — end step also divided by action_repeat
    'horizon_schedule':  f'linear(1, ${{horizon}}, {10000 // ar})',
    'std_schedule':      f'linear(0.5, ${{min_std}}, {10000 // ar})',
    'variance_schedule': f'linear(0.1, 0.001, {20000 // ar})',
    'dec_reward_coeff':  0,
}))

In [5]:
print(cfg.horizon_schedule)
print(cfg.std_schedule)
print(cfg.variance_schedule)

linear(1, 5, 2500)
linear(0.5, 0.05, 2500)
linear(0.1, 0.001, 5000)


In [5]:
from implementation import TDMPC_O2
from algorithm.helper import linear_schedule
from algorithm.helper import Episode, ReplayBuffer
import implementation.logging as l
import time
import torch

In [6]:
import importlib
import implementation
importlib.reload(implementation)

<module 'implementation' from '/content/td-mpc_o2_local/implementation/__init__.py'>

In [25]:
env = make_env(cfg)
agent = TDMPC_O2(cfg)
buffer = ReplayBuffer(cfg)

In [26]:
RESULTS_DIR = '/content/results'

In [27]:
episode_idx, start_time = 0, time.time()
print("\n" + "=" * 50)
print("\033[1m🚀 Training Configuration\033[0m")
print("=" * 50)
for key, value in agent.cfg.items():
    print(f"{key:25}: {value}")
print("=" * 50 + "\n")
save_dir = l.make_save_dir_path(agent.cfg, base_dir = RESULTS_DIR)
print("Saving results to:", save_dir)

all_metrics = []

for step in range(0, cfg.train_steps, cfg.episode_length):

    obs     = env.reset()
    episode = PGEpisode(cfg, obs)       # ← PGEpisode, not Episode
    current_step     = 0
    half_time_reward = 0.0

    episode_start_time = time.time()
        # ---- Rollout -------------------------------------------------------
    while not episode.done:
        if step < cfg.seed_steps:
            action_np = env.action_space.sample()
            action    = torch.tensor(action_np, dtype=torch.float32,
                                      device=agent.device)
        else:
            action, u_mean, u_std, latent_action, log_prob = (
                agent.CEM_in_latent(
                    obs, step=step, t0=episode.first,
                    seed=None, sample_final_action=True,
                )
            )
            # Store on-policy data for this step
            episode.add_pg(log_prob, u_mean, u_std, latent_action)

        obs, reward, done, _ = env.step(action.cpu().numpy())
        current_step        += 1
        if current_step <= 500:
            half_time_reward += reward

        episode += (obs, action, reward, done)
    episode.finalize()
    episode_end_time = time.time()


    buffer+=episode

    horizon = int(linear_schedule(cfg.horizon_schedule,step))
    episode_metrics = {
        'Episode_no:': int(step/cfg.episode_length),
        'Reward': episode.cumulative_reward,
        'Horizon': int(linear_schedule(cfg.horizon_schedule,step)),
        'Std:': linear_schedule(cfg.std_schedule,step),
        'Duration:': episode_end_time - episode_start_time,
    }

    print("Buffer idx:", buffer.idx,", buffer is full:", buffer._full)

    print("\n  Episode Summary")
    print("-" * 25)
    for k, v in episode_metrics.items():
        print(f"  {k:15}: {v}")


    train_metrics = {}
    decoder_metrics = {}


    #     On-policy O2 update
    if step >= cfg.seed_steps:
        t0 = time.time()
        decoder_metrics = update_decoder_pg(agent, episode, step)
        train_metrics["decoder_update_s"] = time.time() - t0

        print("  Decoder Update Metrics:")
        for k, v in decoder_metrics.items():
            print(f"    {k:20}: {v:.4f}")


    #     TD-MPC standard update
    if step >= cfg.seed_steps:
        model_update_start_time = time.time()
        train_metrics = update_tdmpc(agent, buffer, step)
        model_update_end_time = time.time()

    """
    if (step>= cfg.seed_steps):
        decoder_update_start_time = time.time()
        decoder_loss = update_decoder(agent, buffer, cfg, step)
        train_metrics['Decoder_loss'] = decoder_loss
        decoder_update_end_time = time.time()
    """

    print("  Training Metrics:")
    for k, v in train_metrics.items():
        print(f"  {k:15}: {v}")



🚀 Training Configuration
task                     : cheetah-run
modality                 : state
action_repeat            : 4
discount                 : 0.99
episode_length           : 250
train_steps              : 5000
iterations               : 5
num_samples              : 64
num_elites               : 16
mixture_coef             : 0.05
min_std                  : 0.05
temperature              : 0.5
momentum                 : 0.1
batch_size               : 512
max_buffer_size          : 1000000
horizon                  : 5
reward_coef              : 0.5
value_coef               : 0.1
consistency_coef         : 2
rho                      : 0.5
kappa                    : 0.1
lr                       : 0.001
std_schedule             : linear(0.5, 0.05, 2500)
horizon_schedule         : linear(1, 5, 2500)
per_alpha                : 0.6
per_beta                 : 0.4
grad_clip_norm           : 10
seed_steps               : 0
update_freq              : 2
tau                      : 0.01
enc

In [20]:
torch.cuda.empty_cache()

In [7]:
from implementation.episode import PGEpisode

def update_tdmpc(agent, buffer, step):
    """Original TD-MPC update — untouched."""
    agent.model.track_TOLD_grad(True)
    buffer.cfg.batch_size = agent.cfg.batch_size
    metrics = {}
    for i in range(agent.cfg.told_updates):
        update_metrics = agent.update(buffer, step + i)
        for k, v in update_metrics.items():
            metrics[k] = metrics.get(k, 0.0) + v
    for k in metrics:
        metrics[k] /= agent.cfg.told_updates
    return metrics

def update_decoder(agent, buffer, cfg, step):
    """DCEM decoder update."""
    agent.model.track_TOLD_grad(False)
    buffer.cfg.batch_size = agent.cfg.dcem_batch_size
    horizon = int(linear_schedule(cfg.horizon_schedule, step))
    decoder_loss = 0
    for i in range(agent.cfg.decoder_updates):
        obs, *_ = buffer.sample()
        _, u_mean, _, _, _ = agent.DCEMethod(obs, update_mode=True, step=step, t0=False)
        decoder_loss += agent.action_decoder_DDPG_update(obs, u_mean, horizon)
    agent.model.track_TOLD_grad(True)
    return decoder_loss / agent.cfg.decoder_updates

def collect_episode(env, agent, cfg, step):
    """Collect one episode of experience."""
    obs = env.reset()
    episode = Episode(cfg, obs)
    while not episode.done:
        if step < cfg.seed_steps:
            action_np = env.action_space.sample()
            action = torch.tensor(action_np, dtype=torch.float32, device=agent.device)
        else:
            action, _, _, _, _ = agent.CEM_in_latent(
                obs, step=step, t0=episode.first, seed=None, sample_final_action=True
            )
        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    return episode

def update_decoder_pg(agent, episode: PGEpisode, step: int) -> dict:
    """
    Run one full epoch of mini-batch PG updates over the transitions stored
    in *episode*.

    The function:
      1. Freezes TOLD model gradients
      2. Unfreezes decoder + value-network parameters
      3. Iterates over the episode in shuffled mini-batches
      4. Re-enables TOLD model gradients when done

    Args:
        agent:   DCEM_TDMPC (or any agent with .PG_withV, .DCEMethod, .cfg)
        episode: PGEpisode that has already been finalized
        step:    current global environment step (for schedules)

    Returns:
        dict[str, float]: mean of each loss component across all mini-batches
    """
    agent.model.track_TOLD_grad(False)
    agent.model.track_O2_grad(True)

    alpha_v   = linear_schedule(agent.cfg.variance_schedule, step)
    horizon   = int(linear_schedule(agent.cfg.horizon_schedule, step))
    accum: dict[str, list] = {}

    for batch in episode.sample_batches(batch_size=128, shuffle=True):
        (obs_t, action_t, reward_t, obs_t1,
         _old_log, latent_action_t, _umean, _ustd,
         _done, next_rewards, next_obses) = batch

        # Re-run CEM to get fresh distribution parameters (on-policy)
        _, u_mean, u_std, _, _ = agent.DCEMethod(
            obs_t, update_mode=True, step=step, t0=False
        )

        loss_dict = agent.PG_withV(
            obs_t, u_mean, u_std, reward_t, obs_t1,
            latent_action_t, next_rewards, next_obses, alpha_v, horizon,
        )

        for k, v in loss_dict.items():
            accum.setdefault(k, []).append(v)

    agent.model.track_O2_grad(False)
    agent.model.track_TOLD_grad(True)

    return {k: torch.stack(v).mean().item() for k, v in accum.items()}


/content/td-mpc_o2_local


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
%cd /content/td-mpc_o2_local
!git status

#!git config user.email "odysseaskon@gmail.com"
#!git config user.name "odykon"
#!git add .
#!git commit -m "updated files for training"
#!git push https://@github.com/odykon/td-mpc_o2_local.git main


/content/td-mpc_o2_local
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
# search for all uses of action_repeat in the codebase
import subprocess
result = subprocess.run(
    ['grep', '-rn', 'action_repeat', '/content/td-mpc_o2_local/tdmpc'],
    capture_output=True, text=True
)
print(result.stdout)

/content/td-mpc_o2_local/tdmpc/src/logger.py:33:		   ('train steps', f'{int(cfg.train_steps*cfg.action_repeat):,}'),
/content/td-mpc_o2_local/tdmpc/src/env.py:174:	def __init__(self, env, domain, task, action_repeat, modality):
/content/td-mpc_o2_local/tdmpc/src/env.py:210:		self.ep_len = 1000//action_repeat
/content/td-mpc_o2_local/tdmpc/src/env.py:267:	env = ActionRepeatWrapper(env, cfg.action_repeat)
/content/td-mpc_o2_local/tdmpc/src/env.py:279:	env = TimeStepToGymWrapper(env, domain, task, cfg.action_repeat, cfg.modality)
/content/td-mpc_o2_local/tdmpc/src/train.py:77:		env_step = int(step*cfg.action_repeat)
/content/td-mpc_o2_local/tdmpc/cfgs/pixels.yaml:1:train_steps: 3000000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:4:action_repeat: ???
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:6:episode_length: 1000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:7:train_steps: 500000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/tasks/do